# 28e — North West Score-Band Maps with Tunable Left-Side Legend Panels

This notebook generates structural opportunity score-band maps for the North West and subregions.

It uses:

- muted yellow → green score bands;
- a separate left-side legend panel;
- adjustable legend/map spacing controls;
- optional test mode, so one map can be regenerated repeatedly while tuning dimensions;
- blue/cyan borders for wards previously contested by the SDP.

Default outputs are saved to `data/processed/score_band_maps_v3/`.

## 28.1 Setup and paths

Expected project structure:

```text
Electoral_Tribes/
  data/
    processed/
      caveat_resolution_v2/
      target_review_pack_v1/
      sdp_campaign_validation_v2/
      score_band_maps_v1/
    geography/
      boundaries/
        <WD25 ward boundary file>.gpkg/.shp/.geojson
  notebooks/
```

The notebook auto-searches common processed folders and geography folders.

In [10]:
from pathlib import Path
from datetime import datetime
import warnings
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
BOUNDARY_DIR = GEOGRAPHY_DIR

INPUT_DIRS = [
    PROCESSED_DIR / "caveat_resolution_v2",
    PROCESSED_DIR / "report_assets_pre_adam_v1" / "tables",
    PROCESSED_DIR / "report_assets_pre_adam_v1" / "appendices",
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR / "target_model_v2",
    PROCESSED_DIR / "sdp_campaign_validation_v2",
    PROCESSED_DIR / "sdp_campaign_validation_v1",
    PROCESSED_DIR,
    DATA_DIR,
    GEOGRAPHY_DIR,
    NOTEBOOK_DIR,
    PROJECT_DIR,
]

OUTPUT_DIR = PROCESSED_DIR / "score_band_maps_v3"
MAP_DIR = OUTPUT_DIR / "maps"
TABLE_DIR = OUTPUT_DIR / "tables"
MANIFEST_DIR = OUTPUT_DIR / "manifest"

for d in [OUTPUT_DIR, MAP_DIR, TABLE_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

manifest_rows = []


print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)

Project directory: c:\Users\keena\Documents\Electoral_Tribes
Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\score_band_maps_v3


## 28.2 Configuration

The model score is treated as a 0–100 score. If a loaded score column appears to be 0–1, the notebook automatically scales it to 0–100.

In [11]:
NAVY = "#112A46"
BACKGROUND = "#FFFFFF"
OTHER_WARD_FILL = "#F2F2F2"
BOUNDARY_EDGE = "#FFFFFF"
BASE_EDGE = "#C7C7C7"

# Red border is easier to distinguish from the yellow-green score scale.
SDP_BORDER_COLOR = "#E70000"
SDP_BORDER_WIDTH = 1.25

# Muted report-friendly yellow → green scale.
# This avoids making 50–60% wards look visually negative.
SCORE_BANDS = [
    {"label": "Below 40", "lower": -np.inf, "upper": 40, "color": "#D9D9D9"},
    {"label": "40–44.9", "lower": 40, "upper": 45, "color": "#F5F3D7"},
    {"label": "45–49.9", "lower": 45, "upper": 50, "color": "#ECE7A7"},
    {"label": "50–54.9", "lower": 50, "upper": 55, "color": "#D8D982"},
    {"label": "55–59.9", "lower": 55, "upper": 60, "color": "#BDD17A"},
    {"label": "60–64.9", "lower": 60, "upper": 65, "color": "#91C478"},
    {"label": "65–69.9", "lower": 65, "upper": 70, "color": "#5DAE6A"},
    {"label": "70+", "lower": 70, "upper": np.inf, "color": "#247A4A"},
]

SCORE_BAND_ORDER = [b["label"] for b in SCORE_BANDS]
SCORE_BAND_COLORS = {b["label"]: b["color"] for b in SCORE_BANDS}

# -----------------------------
# Layout tuning controls
# -----------------------------
# Use TEST_MODE while adjusting the legend/map gap. This regenerates only one map.
TEST_MODE = False
TEST_REGION = "Greater Manchester"  # North West, Cheshire, Cumbria, Greater Manchester, Lancashire, Merseyside

# These defaults pull the map closer to the legend than the earlier notebook.
DEFAULT_LAYOUT = {
    "figsize": (13.8, 9.2),
    "legend_width_ratio": 0.85,
    "map_width_ratio": 6.0,
    "wspace": -0.08,
    "map_anchor": "W",       # W = anchor map to left/west side of its panel
}

# Quick tuning examples:
# - Map still too far right: legend_width_ratio=0.75, wspace=-0.12
# - Legend too cramped:     legend_width_ratio=1.05, wspace=-0.06
# - Title too cramped:      figsize=(14.5, 9.5)

SUBREGIONS = {
    "cheshire": {
        "label": "Cheshire",
        "lads": ["Cheshire East", "Cheshire West and Chester", "Halton", "Warrington"],
        "figsize": (13.8, 9.2),
        "legend_width_ratio": 0.85,
        "map_width_ratio": 6.0,
        "wspace": -0.08,
        "map_anchor": "W",
    },
    "cumbria": {
        "label": "Cumbria",
        "lads": ["Cumberland", "Westmorland and Furness"],
        "figsize": (13.8, 9.2),
        "legend_width_ratio": 0.85,
        "map_width_ratio": 6.0,
        "wspace": -0.08,
        "map_anchor": "W",
    },
    "greater_manchester": {
        "label": "Greater Manchester",
        "lads": ["Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford", "Stockport", "Tameside", "Trafford", "Wigan"],
        "figsize": (13.8, 9.2),
        "legend_width_ratio": 0.85,
        "map_width_ratio": 6.0,
        "wspace": -0.08,
        "map_anchor": "W",
    },
    "lancashire": {
        "label": "Lancashire",
        "lads": ["Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde", "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley", "Rossendale", "South Ribble", "West Lancashire", "Wyre"],
        "figsize": (13.8, 9.2),
        "legend_width_ratio": 0.85,
        "map_width_ratio": 6.0,
        "wspace": -0.08,
        "map_anchor": "W",
    },
    "merseyside": {
        "label": "Merseyside",
        "lads": ["Knowsley", "Liverpool", "Sefton", "St. Helens", "St Helens", "Wirral"],
        "figsize": (13.8, 9.2),
        "legend_width_ratio": 0.85,
        "map_width_ratio": 6.0,
        "wspace": -0.08,
        "map_anchor": "W",
    },
}


## 28.3 Utility functions

In [12]:
def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("&", "and")
    for ch in [".", ",", "'", "’", "-", "/", "(", ")"]:
        s = s.replace(ch, " ")
    return " ".join(s.split())


def standardise_code(series):
    return series.astype("string").str.strip()


def find_file(filename, required=True):
    for folder in INPUT_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    if PROCESSED_DIR.exists():
        matches = sorted(PROCESSED_DIR.rglob(filename), key=lambda p: p.stat().st_mtime, reverse=True)
        if matches:
            return matches[0]
    sandbox = Path("/mnt/data") / filename
    if sandbox.exists():
        return sandbox
    if required:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    return None


def read_csv(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional missing:", filename)
        return None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df


def first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def score_to_band(value):
    if pd.isna(value):
        return "No score"
    v = float(value)
    for band in SCORE_BANDS:
        if v >= band["lower"] and v < band["upper"]:
            return band["label"]
    return "No score"


def wrap_labels(labels, width=24):
    return ["\n".join(textwrap.wrap(str(label), width=width)) for label in labels]



def add_manifest(filename, asset_type, description, path):
    manifest_rows.append({
        "filename": filename,
        "asset_type": asset_type,
        "description": description,
        "path": str(path),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    })


def save_table(df, filename, description):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    add_manifest(filename, "table_csv", description, path)
    print("Saved table:", path)
    return path


def save_map(fig, filename, description):
    path = MAP_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor=BACKGROUND)
    plt.close(fig)
    add_manifest(filename, "map_png", description, path)
    print("Saved map:", path)
    return path

## 28.4 Load model rows, SDP rows and boundaries

In [13]:
# Prefer the corrected caveat-resolution/reportable file.
MODEL_CANDIDATES = [
    "north_west_reportable_main_review_v2.csv",
    "north_west_revised_consolidated_review_v2.csv",
    "pre_adam_party_transition_diagnostics_all_v4.csv",
    "north_west_consolidated_target_review_v1.csv",
    "all_available_consolidated_target_review_v1.csv",
]

model = None
model_source = None
for fname in MODEL_CANDIDATES:
    try:
        df = read_csv(fname, required=False)
        if df is not None and len(df) > 0:
            if "WD25CD" in df.columns or "WD25CD_model" in df.columns or "WD25CD_final" in df.columns:
                model = df.copy()
                model_source = fname
                break
    except Exception as e:
        print("Could not load", fname, e)

if model is None:
    raise FileNotFoundError("Could not find a usable model/review file with WD25CD codes.")

print("Using model source:", model_source)
print("Model rows:", len(model))

# SDP contested wards.
sdp = read_csv("sdp_campaign_wards_profile_v2.csv", required=False)
if sdp is None:
    sdp = read_csv("sdp_campaign_wards_profile_v1.csv", required=False)

print("SDP source rows:", 0 if sdp is None else len(sdp))

Loaded north_west_reportable_main_review_v2.csv: (824, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_reportable_main_review_v2.csv
Using model source: north_west_reportable_main_review_v2.csv
Model rows: 824
Loaded sdp_campaign_wards_profile_v2.csv: (171, 64) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v2\sdp_campaign_wards_profile_v2.csv
SDP source rows: 171


In [14]:
def find_boundary_file():
    search_dirs = [BOUNDARY_DIR, GEOGRAPHY_DIR, DATA_DIR, PROJECT_DIR, Path("/mnt/data")]
    patterns = ["*.gpkg", "*.shp", "*.geojson", "*.json"]
    candidates = []
    for folder in search_dirs:
        if folder.exists():
            for pattern in patterns:
                candidates.extend(folder.rglob(pattern))
    if not candidates:
        return None
    preferred = [p for p in candidates if any(token in p.name.lower() for token in ["ward", "wd25", "may_2025", "december_2024"])]
    return preferred[0] if preferred else candidates[0]

boundary_file = find_boundary_file()
print("Boundary file:", boundary_file)

if boundary_file is None:
    raise FileNotFoundError("No boundary file found. Put a WD25 ward boundary .gpkg/.shp/.geojson in data/geography/boundaries or data/geography.")

try:
    import geopandas as gpd
except Exception as e:
    raise ImportError("geopandas is required for map generation.") from e

wards = gpd.read_file(boundary_file)
print("Boundary rows:", len(wards))
print("Boundary columns:", wards.columns.tolist())

if "WD25CD" not in wards.columns:
    wd_candidates = [c for c in wards.columns if c.upper() == "WD25CD" or "WD25CD" in c.upper()]
    if wd_candidates:
        wards = wards.rename(columns={wd_candidates[0]: "WD25CD"})
    else:
        raise KeyError("Boundary file does not contain WD25CD or a recognisable equivalent.")

wards["WD25CD"] = standardise_code(wards["WD25CD"])

try:
    wards = wards.to_crs(27700)
except Exception:
    pass

Boundary file: c:\Users\keena\Documents\Electoral_Tribes\data\geography\Wards_May_2025_Boundaries_UK_BGC.gpkg
Boundary rows: 8405
Boundary columns: ['WD25CD', 'WD25NM', 'WD25NMW', 'LAD25CD', 'LAD25NM', 'LAD25NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']


## 28.5 Prepare score bands and SDP contested flags

In [15]:
# Standardise model ward code.
code_col = first_existing_col(model, ["WD25CD", "WD25CD_model", "WD25CD_final", "ward_code"])
if code_col is None:
    raise KeyError(f"No WD25 code column found in model. Columns: {model.columns.tolist()}")

model["WD25CD"] = standardise_code(model[code_col])

# Score column.
score_col = first_existing_col(model, [
    "initial_watchlist_score", "model_score", "structural_opportunity_score", "score", "target_score"
])
if score_col is None:
    raise KeyError(f"No score column found. Columns: {model.columns.tolist()}")

model["score_for_map"] = pd.to_numeric(model[score_col], errors="coerce")
if model["score_for_map"].dropna().max() <= 1.5:
    model["score_for_map"] = model["score_for_map"] * 100

# Name columns.
if "LAD25NM" not in model.columns:
    model["LAD25NM"] = model.get("LAD25NM_model", "")
if "WD25NM" not in model.columns:
    model["WD25NM"] = model.get("WD25NM_model", model.get("ward_name", ""))

model["score_band"] = model["score_for_map"].map(score_to_band)
model["score_band_order"] = pd.Categorical(model["score_band"], categories=SCORE_BAND_ORDER + ["No score"], ordered=True)
model["score_band_color"] = model["score_band"].map(SCORE_BAND_COLORS).fillna("#F7F7F7")
model["clean_lad"] = model["LAD25NM"].map(clean_text)

# If the selected model is all-region, keep North West LADs only.
all_subregion_lads = {clean_text(x) for cfg in SUBREGIONS.values() for x in cfg["lads"]}
model_nw = model[model["clean_lad"].isin(all_subregion_lads)].copy()
if len(model_nw) == 0:
    # Fallback: if file is already North West but LAD names were unusual, keep all rows.
    print("Warning: no rows matched configured North West LAD names. Keeping all model rows.")
    model_nw = model.copy()

# SDP contested wards.
sdp_codes = set()
if sdp is not None and len(sdp) > 0:
    sdp_code_col = first_existing_col(sdp, ["WD25CD_model", "WD25CD_final", "WD25CD", "ward_code"])
    if sdp_code_col is not None:
        sdp_codes = set(standardise_code(sdp[sdp_code_col]).dropna())
    else:
        print("No SDP WD25 code column found; SDP borders will not be drawn.")

model_nw["sdp_contested"] = model_nw["WD25CD"].isin(sdp_codes)

print("North West model rows:", len(model_nw))
print("Score column:", score_col)
print("SDP contested model wards:", int(model_nw["sdp_contested"].sum()))
print(model_nw["score_band"].value_counts(dropna=False).reindex(SCORE_BAND_ORDER + ["No score"]))

North West model rows: 824
Score column: initial_watchlist_score
SDP contested model wards: 6
score_band
Below 40     87.0
40–44.9      82.0
45–49.9     133.0
50–54.9     140.0
55–59.9     163.0
60–64.9     118.0
65–69.9      75.0
70+          26.0
No score      NaN
Name: count, dtype: float64


## 28.6 Join model rows to boundaries

In [16]:
# Prepare a boundary layer that will not create _x/_y suffixes on LAD/ward name columns.
# Some WD25 boundary files already contain LAD25NM/WD25NM. If these are left in the
# boundary GeoDataFrame, the merge creates LAD25NM_x / LAD25NM_y and later code cannot
# find a plain LAD25NM column.
boundary_name_cols = [
    "WD25NM", "WD25NM_x", "WD25NM_y", "WD25NM_model",
    "LAD25NM", "LAD25NM_x", "LAD25NM_y", "LAD25NM_model",
    "LAD25CD", "LAD25CD_x", "LAD25CD_y", "LAD25CD_model",
]
wards_for_merge = wards.drop(columns=[c for c in boundary_name_cols if c in wards.columns], errors="ignore").copy()

model_map_cols = ["WD25CD", "WD25NM", "LAD25NM", "score_for_map", "score_band", "score_band_color", "sdp_contested"]
missing_model_cols = [c for c in model_map_cols if c not in model_nw.columns]
if missing_model_cols:
    raise KeyError(f"Model file is missing required map columns: {missing_model_cols}. Available columns: {model_nw.columns.tolist()}")

map_gdf = wards_for_merge.merge(
    model_nw[model_map_cols].drop_duplicates("WD25CD"),
    on="WD25CD",
    how="inner"
)

if len(map_gdf) == 0:
    raise ValueError("No boundary rows matched the model WD25CD codes.")

# Defensive repair in case any future boundary source still causes suffixes.
def ensure_plain_column(df, plain_name, fallback_candidates):
    if plain_name in df.columns:
        return df
    for candidate in fallback_candidates:
        if candidate in df.columns:
            df[plain_name] = df[candidate]
            return df
    raise KeyError(f"Could not create {plain_name}. Available columns: {df.columns.tolist()}")

map_gdf = ensure_plain_column(map_gdf, "LAD25NM", ["LAD25NM_model", "LAD25NM_y", "LAD25NM_x"])
map_gdf = ensure_plain_column(map_gdf, "WD25NM", ["WD25NM_model", "WD25NM_y", "WD25NM_x", "ward_name"])

map_gdf["clean_lad"] = map_gdf["LAD25NM"].map(clean_text)
print("Mapped rows:", len(map_gdf))
print("Mapped SDP-contested rows:", int(map_gdf["sdp_contested"].sum()))
print("Map columns include LAD25NM:", "LAD25NM" in map_gdf.columns)

ward_index = map_gdf.drop(columns="geometry").copy()
save_table(ward_index, "map_28_score_band_ward_index_v1.csv", "Ward-level score band and SDP-contested flag used for score-band maps.")

Mapped rows: 824
Mapped SDP-contested rows: 6
Map columns include LAD25NM: True
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\score_band_maps_v3\tables\map_28_score_band_ward_index_v1.csv


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/score_band_maps_v3/tables/map_28_score_band_ward_index_v1.csv')

## 28.7 Map plotting functions

In [17]:
def wrap_label(text, width=24):
    return "\n".join(textwrap.wrap(str(text), width=width))


def plot_score_band_map(
    gdf,
    title,
    subtitle,
    filename,
    description,
    figsize=None,
    legend_width_ratio=None,
    map_width_ratio=None,
    wspace=None,
    map_anchor=None,
):
    """Plot a score-band map with a separate, tunable left-side legend panel."""
    if len(gdf) == 0:
        print("Skipping empty map:", title)
        return None

    # Pull layout settings from DEFAULT_LAYOUT unless overridden per map.
    figsize = figsize or DEFAULT_LAYOUT["figsize"]
    legend_width_ratio = legend_width_ratio if legend_width_ratio is not None else DEFAULT_LAYOUT["legend_width_ratio"]
    map_width_ratio = map_width_ratio if map_width_ratio is not None else DEFAULT_LAYOUT["map_width_ratio"]
    wspace = wspace if wspace is not None else DEFAULT_LAYOUT["wspace"]
    map_anchor = map_anchor or DEFAULT_LAYOUT["map_anchor"]

    plot_gdf = gdf.copy()
    plot_gdf["score_band"] = (
        plot_gdf["score_band"]
        .astype("string")
        .fillna("No score")
        .astype(str)
    )
    plot_gdf["score_band"] = pd.Categorical(
        plot_gdf["score_band"],
        categories=SCORE_BAND_ORDER + ["No score"],
        ordered=True,
    )
    plot_gdf = plot_gdf.sort_values("score_band")
    plot_gdf["_plot_colour"] = plot_gdf["score_band"].astype(str).map(SCORE_BAND_COLORS).fillna(OTHER_WARD_FILL)

    fig = plt.figure(figsize=figsize)
    fig.patch.set_facecolor(BACKGROUND)

    gs = fig.add_gridspec(
        nrows=1,
        ncols=2,
        width_ratios=[legend_width_ratio, map_width_ratio],
        wspace=wspace,
    )

    legend_ax = fig.add_subplot(gs[0, 0])
    map_ax = fig.add_subplot(gs[0, 1])
    legend_ax.set_facecolor(BACKGROUND)
    map_ax.set_facecolor(BACKGROUND)

    # Base layer first, to preserve a visible geography outline.
    plot_gdf.plot(
        ax=map_ax,
        color=OTHER_WARD_FILL,
        edgecolor=BASE_EDGE,
        linewidth=0.12,
    )

    # Plot each score band separately so the legend order is controlled.
    for band in SCORE_BAND_ORDER:
        sub = plot_gdf[plot_gdf["score_band"].astype(str).eq(band)]
        if len(sub) == 0:
            continue
        sub.plot(
            ax=map_ax,
            color=SCORE_BAND_COLORS[band],
            edgecolor=BOUNDARY_EDGE,
            linewidth=0.16,
        )

    # SDP-contested border overlay.
    sdp_sub = plot_gdf[plot_gdf["sdp_contested"].fillna(False).astype(bool)]
    if len(sdp_sub) > 0:
        sdp_sub.boundary.plot(
            ax=map_ax,
            color=SDP_BORDER_COLOR,
            linewidth=SDP_BORDER_WIDTH,
        )

    map_ax.set_axis_off()
    map_ax.set_aspect("equal")
    map_ax.set_anchor(map_anchor)  # Key gap-control line: anchor geometry to left side of map panel.
    map_ax.set_title(title, fontsize=22, fontweight="bold", color=NAVY, pad=16)
    if subtitle:
        map_ax.text(
            0.5,
            0.985,
            subtitle,
            transform=map_ax.transAxes,
            ha="center",
            va="top",
            fontsize=10,
            color="#4A4A4A",
        )

    # Dedicated legend panel.
    legend_ax.set_axis_off()
    legend_ax.text(
        0.02,
        0.96,
        "Structural opportunity score",
        ha="left",
        va="top",
        fontsize=15,
        fontweight="bold",
        color=NAVY,
        transform=legend_ax.transAxes,
    )

    legend_ax.text(
        0.02,
        0.79,
        "Wards grouped into 5-point model-score bands.",
        ha="left",
        va="top",
        fontsize=9,
        color="#4A4A4A",
        transform=legend_ax.transAxes,
    )

    handles = []
    for band in SCORE_BAND_ORDER:
        if (plot_gdf["score_band"].astype(str) == band).any():
            handles.append(
                Patch(
                    facecolor=SCORE_BAND_COLORS[band],
                    edgecolor="white",
                    label=wrap_label(band, 22),
                )
            )

    if len(sdp_sub) > 0:
        handles.append(
            Patch(
                facecolor="white",
                edgecolor=SDP_BORDER_COLOR,
                linewidth=SDP_BORDER_WIDTH,
                label="SDP previously contested",
            )
        )

    legend_ax.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.02, 0.42),
        frameon=False,
        fontsize=9,
        handlelength=1.5,
        handleheight=1.1,
        labelspacing=0.9,
        borderaxespad=0,
    )

    # Small note in the legend panel.
    legend_ax.text(
        0.02,
        0.08,
        "Note: this is not a vote-share\nforecast or seat-win probability.",
        ha="left",
        va="bottom",
        fontsize=8,
        color="#4A4A4A",
        transform=legend_ax.transAxes,
    )

    print(
        f"Layout for {title}: figsize={figsize}, legend_width_ratio={legend_width_ratio}, "
        f"map_width_ratio={map_width_ratio}, wspace={wspace}, map_anchor={map_anchor}"
    )
    return save_map(fig, filename, description)


def get_subregion_gdf(gdf, key):
    lads = {clean_text(x) for x in SUBREGIONS[key]["lads"]}
    return gdf[gdf["clean_lad"].isin(lads)].copy()


## 28.8 Generate North West and subregion maps

In [18]:
# Build map queue.
REGIONS_TO_PLOT = {
    "North West": {
        "gdf": map_gdf,
        "title": "North West structural opportunity score",
        "subtitle": "Wards grouped into 5-point score bands; red border marks previously SDP-contested wards.",
        "filename": "map_28_north_west_score_bands_sdp_borders_v3.png",
        "description": "Full North West ward map using structural opportunity/model score bands with SDP-contested ward borders and a tuned left-side legend.",
        **DEFAULT_LAYOUT,
    }
}

for key, cfg in SUBREGIONS.items():
    REGIONS_TO_PLOT[cfg["label"]] = {
        "gdf": get_subregion_gdf(map_gdf, key),
        "title": f"{cfg['label']} structural opportunity score",
        "subtitle": "Score bands use the same North West scale; red border marks previously SDP-contested wards.",
        "filename": f"map_28_{key}_score_bands_sdp_borders_v3.png",
        "description": f"{cfg['label']} zoom map using structural opportunity/model score bands with SDP-contested ward borders and a tuned left-side legend.",
        "figsize": cfg.get("figsize", DEFAULT_LAYOUT["figsize"]),
        "legend_width_ratio": cfg.get("legend_width_ratio", DEFAULT_LAYOUT["legend_width_ratio"]),
        "map_width_ratio": cfg.get("map_width_ratio", DEFAULT_LAYOUT["map_width_ratio"]),
        "wspace": cfg.get("wspace", DEFAULT_LAYOUT["wspace"]),
        "map_anchor": cfg.get("map_anchor", DEFAULT_LAYOUT["map_anchor"]),
    }

if TEST_MODE:
    if TEST_REGION not in REGIONS_TO_PLOT:
        raise KeyError(f"TEST_REGION '{TEST_REGION}' not recognised. Options: {list(REGIONS_TO_PLOT)}")
    print(f"TEST_MODE active: plotting only {TEST_REGION}")
    REGIONS_TO_PLOT = {TEST_REGION: REGIONS_TO_PLOT[TEST_REGION]}

for label, cfg in REGIONS_TO_PLOT.items():
    plot_score_band_map(
        cfg["gdf"],
        title=cfg["title"],
        subtitle=cfg["subtitle"],
        filename=cfg["filename"],
        description=cfg["description"],
        figsize=cfg.get("figsize"),
        legend_width_ratio=cfg.get("legend_width_ratio"),
        map_width_ratio=cfg.get("map_width_ratio"),
        wspace=cfg.get("wspace"),
        map_anchor=cfg.get("map_anchor"),
    )


Layout for North West structural opportunity score: figsize=(13.8, 9.2), legend_width_ratio=0.85, map_width_ratio=6.0, wspace=-0.08, map_anchor=W
Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\score_band_maps_v3\maps\map_28_north_west_score_bands_sdp_borders_v3.png
Layout for Cheshire structural opportunity score: figsize=(13.8, 9.2), legend_width_ratio=0.85, map_width_ratio=6.0, wspace=-0.08, map_anchor=W
Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\score_band_maps_v3\maps\map_28_cheshire_score_bands_sdp_borders_v3.png
Layout for Cumbria structural opportunity score: figsize=(13.8, 9.2), legend_width_ratio=0.85, map_width_ratio=6.0, wspace=-0.08, map_anchor=W
Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\score_band_maps_v3\maps\map_28_cumbria_score_bands_sdp_borders_v3.png
Layout for Greater Manchester structural opportunity score: figsize=(13.8, 9.2), legend_width_ratio=0.85, map_width_ratio=6.0, wspace=-0.08, map_anch

## 28.9 Export subregion summary and manifest

In [19]:
summary_rows = []
for key, cfg in SUBREGIONS.items():
    sub = get_subregion_gdf(map_gdf, key)
    row = {
        "subregion_key": key,
        "subregion_label": cfg["label"],
        "wards": len(sub),
        "sdp_contested_wards": int(sub["sdp_contested"].sum()) if len(sub) else 0,
        "mean_score": float(sub["score_for_map"].mean()) if len(sub) else np.nan,
        "max_score": float(sub["score_for_map"].max()) if len(sub) else np.nan,
    }
    for band in SCORE_BAND_ORDER:
        row[f"score_band_{band}_wards"] = int((sub["score_band"].astype(str) == band).sum()) if len(sub) else 0
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
save_table(summary, "map_28_score_band_summary_by_subregion_v1.csv", "Subregion summary of score bands and SDP-contested wards.")

manifest = pd.DataFrame(manifest_rows)
manifest_path = MANIFEST_DIR / "map_28_score_band_maps_manifest_v1.csv"
manifest.to_csv(manifest_path, index=False)
print("Saved manifest:", manifest_path)
manifest

Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\score_band_maps_v3\tables\map_28_score_band_summary_by_subregion_v1.csv
Saved manifest: c:\Users\keena\Documents\Electoral_Tribes\data\processed\score_band_maps_v3\manifest\map_28_score_band_maps_manifest_v1.csv


,filename,asset_type,description,path,created_at
0,map_28_score_band_ward_index_v1.csv,table_csv,Ward-level score band and SDP-contested flag u...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:18
1,map_28_north_west_score_bands_sdp_borders_v3.png,map_png,Full North West ward map using structural oppo...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:26
2,map_28_cheshire_score_bands_sdp_borders_v3.png,map_png,Cheshire zoom map using structural opportunity...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:28
3,map_28_cumbria_score_bands_sdp_borders_v3.png,map_png,Cumbria zoom map using structural opportunity/...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:30
4,map_28_greater_manchester_score_bands_sdp_bord...,map_png,Greater Manchester zoom map using structural o...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:31
5,map_28_lancashire_score_bands_sdp_borders_v3.png,map_png,Lancashire zoom map using structural opportuni...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:34
6,map_28_merseyside_score_bands_sdp_borders_v3.png,map_png,Merseyside zoom map using structural opportuni...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:35
7,map_28_score_band_summary_by_subregion_v1.csv,table_csv,Subregion summary of score bands and SDP-conte...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-29T09:32:35


## 28.10 Expected outputs

The generated maps are saved to:

```text
data/processed/score_band_maps_v3/maps/
```

Expected files:

```text
map_28_north_west_score_bands_sdp_borders_v3.png
map_28_cheshire_score_bands_sdp_borders_v3.png
map_28_cumbria_score_bands_sdp_borders_v3.png
map_28_greater_manchester_score_bands_sdp_borders_v3.png
map_28_lancashire_score_bands_sdp_borders_v3.png
map_28_merseyside_score_bands_sdp_borders_v3.png
```

For fast layout tuning, set:

```python
TEST_MODE = True
TEST_REGION = "Greater Manchester"
```

Then adjust `DEFAULT_LAYOUT` or the relevant entry in `SUBREGIONS` until the legend/map gap looks right. Once satisfied, set `TEST_MODE = False` and rerun the notebook to regenerate all maps.